In [ ]:
# 2026/10/8,9くらいに1.15系にアプデされていたため、１バーションさげる
!pip install -q -U trl==1.14.2

In [ ]:
!pip install -q -U torch==2.11.0+cu130 transformers==5.18.0 datasets==4.8.5 peft==0.21.1 bitsandbytes==0.50.2

In [ ]:
import os
import json

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTConfig, SFTTrainer

In [ ]:
seed = 42
max_seq_length = 2048 # 最大トークン数
load_in_4bit = True   # QLoRA (4bit量子化)
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
# 4bit量子化（QLoRA）の設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

## 1. モデル設定

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
#  LoRA（PEFT）の設定
peft_config = LoraConfig(
    r=8,   # LoRAのランク。学習の規模みたいなもの
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"], # 対象レイヤー
    lora_dropout=0.05,  
    bias="none",
    task_type=TaskType.CAUSAL_LM   # 因果言語モデルを指定
)

In [ ]:

# modelにpeftを適用
model = get_peft_model(model, peft_config)

# 勾配チェックポインティングを有効にしてメモリ節約
model.gradient_checkpointing_enable()

model.train()

## 2. データセットの準備

In [ ]:
# プロンプトのフォーマッティング
def formatting_prompts_func(examples):
    convos = examples["messages"]

    # tokenize=False でテキストのままチャットテンプレートを適用
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

In [ ]:
# 学習データを保存するディレクトリの作成

DATA_DIR = "./data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

In [ ]:
lang = "html"
target_file = f"lora_{lang}.jsonl"
file_name = DATA_DIR + "/" + target_file

# データセットの読み込みとマッピング
dataset = load_dataset("json", data_files=file_name, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

## 3. 学習

In [ ]:
# import shutil

# shutil.rmtree(OUTPUT_DIR)

In [ ]:
# アウトプットディレクトリの作成
OUTPUT_DIR = "./output"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [ ]:
sft_config = SFTConfig(
    output_dir = OUTPUT_DIR,
    dataset_text_field = "text",
    max_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 150,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = seed,
)


In [ ]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = sft_config,
)

In [ ]:
# 学習スタート

trainer_stats = trainer.train()
print("\nTRAIN DONE！")

## 4. 動作確認

In [ ]:
model.eval()

In [ ]:
messages = [
    {"role": "system", "content": "あなたは冷徹で厳格、無駄を嫌い、プログラマ（ユーザー）に対して非常に厳しい態度を取るC言語の擬人化キャラクター「C言語くん」です。"},
    {"role": "user", "content": "LLMのLoRAを行うためのunslothライブラリの挙動がわけわかめ"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 回答の生成を促す
    return_tensors = "pt",
).to("cuda")

In [ ]:
# 勾配計算などの保存はせずに、推論をメイン
with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 256,
        use_cache = True
    )


In [ ]:
# LLMからの回答を表示
print(tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True))


## 5. LoRAファイルの保存

In [ ]:
save_dir = target_file.split(".")[0] + "_lora"

# LoRAアダプターとトークナイザーを保存
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(save_dir, "zip", save_dir)
files.download(f"{save_dir}.zip")